# DQ STG BS

Визуализация последнего прогона `dq-fct-bs` (логи `DQ_FCT_BS`), сводка parquet `fct_bs` и карта точек активных БС (folium).

Перед запуском: `uv run mobile build-fct-bs`, затем `uv run mobile dq-fct-bs`.

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

from mobile.project_paths import PROJECT_ROOT

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)

ROOT = PROJECT_ROOT
TAG = "DQ_FCT_BS"
BOUNDARY = "dataset_basic"

In [ ]:
from mobile.pipelines.nb.common import checks_by_status, load_dq_dashboard

try:
    dq_logs, latest, meta = load_dq_dashboard(ROOT, tag=TAG, boundary_check=BOUNDARY)
except (FileNotFoundError, ValueError) as exc:
    print(f"Нет DQ-логов для {TAG}: {exc}")
    latest = meta = None
else:
    print("Последний прогон:", meta)
    display(checks_by_status(latest))

## Метрики DQ (последний прогон)

Графики строятся только из полей `metrics` в логах `DQ_FCT_BS`.

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import (
    render_fct_bs_dq_cardinality,
    render_fct_bs_dq_gates,
    render_fct_bs_dq_geometry_detail,
    render_fct_bs_dq_nulls,
    render_fct_bs_dq_overview,
)

if latest is None:
    print("Нет данных для визуализации")
else:
    for renderer in (
        render_fct_bs_dq_overview,
        render_fct_bs_dq_gates,
        render_fct_bs_dq_nulls,
        render_fct_bs_dq_cardinality,
        render_fct_bs_dq_geometry_detail,
    ):
        fig = renderer(latest)
        plt.show()

In [ ]:
from mobile.pipelines.nb.common import failed_warning_table, metrics_wide_table

if latest is None:
    pass
else:
    display(pd.DataFrame([meta]))
    print("\n--- failed / warning ---")
    display(failed_warning_table(latest))
    print("\n--- все метрики (плоская таблица) ---")
    wide = metrics_wide_table(latest)
    display(wide.head(100))
    if len(wide) > 100:
        print(f"... всего строк metrics: {len(wide)}")

## Профиль parquet `fct_bs`

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import display_fct_bs_parquet_summary, render_fct_bs_parquet_scd_mix

try:
    display_fct_bs_parquet_summary(ROOT)
except FileNotFoundError as exc:
    print(exc)
else:
    fig = render_fct_bs_parquet_scd_mix(ROOT)
    plt.show()

## Карта `fct_bs` (folium)

Точки **lon/lat** (FastMarkerCluster). **`uv run mobile nb-fct-bs`:** только кластер. **В Jupyter:** `display_fct_bs_folium_maps(ROOT, interactive=True)` — фильтры **mnc** / **telecomstandard** / **bs_type**. Активные БС (`date_off = 2262-04-11`).

In [ ]:
from mobile.pipelines.nb.common import display_fct_bs_folium_maps

try:
    display_fct_bs_folium_maps(ROOT, active_only=True)
except (FileNotFoundError, ValueError) as exc:
    print(exc)